# DATASCAPE 2026 — Bukti Audit Dataset

Notebook ini menggabungkan laporan audit dan kode audit mandiri. Seluruh kode tersedia di dalam `.ipynb`; tidak membutuhkan file `.py`.

**Bagian pertama adalah hasil audit historis 1 Oktober 2026**, bukan hasil eksekusi runtime Anda. Jalankan bagian reproduksi untuk menghitung ulang dari dataset yang dipilih. Tidak ada training, pengubahan label, penghapusan data, atau prediksi test.

Untuk konteks bisnis, aturan lomba, AI Declaration, visualisasi, dan lembar keputusan, lihat notebook `datascape_2026_data_understanding.ipynb`.

**AI Declaration:** kode dan dokumentasi disusun dengan bantuan OpenAI Codex. Tidak menggunakan AI/LLM/SLM/VLM untuk menghasilkan label atau prediksi gambar kompetisi. Tim harus menyesuaikan deklarasi dengan penggunaan AI aktual sebelum pengumpulan.

# Audit dataset DATASCAPE — 1 Oktober 2026

Audit dilakukan langsung pada dataset lokal, bukan hanya membaca notebook.
Seluruh 12.224 gambar training diperiksa secara otomatis. Data test hanya
diperiksa nama file dan kecocokannya dengan template, tanpa inspeksi piksel
atau penentuan label test. Tidak ada gambar atau label yang diubah.

## Atribut yang terbukti tersedia

- Label training berasal dari direktori `Train_Set/Aman`, `Rusak`, dan `Lainnya`.
- Identitas gambar berasal dari nama dan lokasi file; tidak ada CSV metadata training.
- Satu-satunya CSV dataset adalah `sample_submission.csv`, dengan kolom
  `filename,label` dan 3.057 baris. Semua label template berisi `Rusak`;
  ini **bukan ground truth test**.
- Seluruh 3.057 nama gambar test cocok dengan template; tidak ada ID duplikat,
  file yang hilang, atau file tambahan pada pemeriksaan nama gambar.
- Atribut turunan yang diekstrak: format aktual, mode warna, ukuran file,
  lebar, tinggi, rasio aspek, kecerahan, kontras, proporsi piksel gelap/terang,
  proksi ketajaman, serta SHA-256 file dan piksel RGB setelah orientasi EXIF.
- Sebanyak 202 gambar training yang terbaca memiliki metadata EXIF. Isi tag
  tidak dianalisis; keberadaan EXIF tidak membuktikan adanya waktu/lokasi pengiriman.
- Tidak ditemukan tabel atribut kurir, lokasi, waktu pengiriman, bahan kemasan,
  jenis cacat, atau tingkat keparahan. Informasi semantik yang mungkin terlihat
  pada gambar belum dianotasi oleh audit ini.

## Temuan terukur

| Kelas | File training | Persentase | Median lebar / tinggi | Sisi pendek <224 piksel |
|---|---:|---:|---:|---:|
| Aman | 8.800 | 71,99% | 512 / 512 | 545 (6,19% dari gambar terbaca) |
| Rusak | 3.001 | 24,55% | 640 / 640 | 140 (4,67%) |
| Lainnya | 423 | 3,46% | 200 / 200 | 301 (71,16%) |

Median lebar dan tinggi dihitung terpisah, bukan ukuran semua gambar kelas.
Statistik piksel menggunakan 12.223 gambar yang berhasil dibaca.

1. **Satu gambar gagal dibaca:** `Train_Set/Aman/Aman_06692.jpg` menghasilkan
   `UnidentifiedImageError`. File dipertahankan; perlu dikarantina dari pipeline
   training atau diganti dari sumber resmi setelah pengecekan.
2. **Duplikasi persis:** 607 kelompok berisi 1.260 gambar, setara 653 salinan
   berlebih jika setiap kelompok dipertahankan satu kali. Pemeriksaan byte file
   dan piksel menghasilkan jumlah yang sama. Gambar yang hampir mirip belum diperiksa.
3. **Konflik label:** 26 kelompok duplikat persis memiliki lebih dari satu label.
   Contoh pasangan identik: `Aman_07814.jpg` dengan `Rusak_02189.jpg`,
   `Aman_06655.jpg` dengan `Rusak_01870.jpg`, dan
   `Aman_07808.jpg` dengan `Rusak_02115.jpg`.
   Audit membuktikan inkonsistensi, tetapi tidak menentukan label mana yang benar.
4. **Kebocoran validasi baseline:** reproduksi urutan discovery notebook pada
   filesystem saat audit, dengan split stratified 80/20 dan seed 42 setelah
   mengecualikan file gagal baca, menghasilkan 9.778 training dan 2.445 validation.
   Ada 198 kelompok piksel identik di kedua sisi; **199 gambar validation
   (8,14%) identik dengan gambar training**. Pemeriksaan notebook hanya membandingkan
   path, sehingga lolos meskipun isinya identik. Angka split bergantung urutan discovery;
   jumlah duplikat dataset tidak. Belum diukur seberapa besar pengaruhnya pada skor model.
5. **Perbedaan kualitas/resolusi antarkelas:** kelas Lainnya jauh lebih sering
   beresolusi rendah. Median brightness Aman 0,603; Rusak 0,541; Lainnya 0,557.
   Ini indikasi perbedaan distribusi akuisisi gambar, bukan bukti penyebab kerusakan
   atau bukti bahwa model telah menggunakan ciri tersebut.
6. **Kandidat audit kualitas:** 235 gambar memiliki mean brightness <0,2;
   2.019 >0,8; 986 memiliki sisi pendek <224 piksel; 206 memiliki rasio aspek
   di luar 0,5–2. Ambang ini eksploratif. Latar putih dapat meningkatkan kecerahan;
   skor ketajaman juga dipengaruhi tekstur. Tidak otomatis berarti foto gagal atau paket rusak.
7. **Format aktual:** 10.757 JPEG, 1.371 WEBP, dan 95 PNG pada gambar terbaca.
   Format dideteksi dari isi file, bukan ekstensi. Mode warna meliputi RGB (12.195),
   RGBA (21), dan palet/P (7), sehingga preprocessing perlu konsisten.

## Insight dan pekerjaan selain klasifikasi

| Prioritas | Analisis | Bukti pendukung dan hasil yang dituju |
|---|---|---|
| 1 | Audit konsistensi label dan deduplikasi | 26 kelompok konflik; buat antrean tinjauan training, tanpa memilih label otomatis. |
| 2 | Audit kebocoran evaluasi | Kelompokkan gambar identik sebelum split; konflik label ditinjau atau dikarantina dahulu. Lanjutkan deteksi near-duplicate. |
| 3 | Analisis kualitas akuisisi foto | Ukuran, kecerahan, kontras, dan ketajaman dapat menjadi dasar antrean pemeriksaan kualitas foto. |
| 4 | Analisis bias sumber gambar | Bandingkan distribusi kualitas tiap kelas; evaluasi performa menurut resolusi/pencahayaan setelah validation dibersihkan. |
| 5 | Clustering dan pencarian gambar serupa | Eksplorasi kelompok visual dan contoh mirip pada training yang sudah dideduplikasi. Belum dijalankan; cluster perlu interpretasi. |
| 6 | Deteksi anomali | Prioritaskan foto tidak lazim untuk tinjauan; anomali visual tidak otomatis kerusakan paket. Belum dijalankan. |

Lokalisasi cacat, tipe cacat, dan skor keparahan membutuhkan anotasi tambahan
untuk evaluasi yang dapat dipercaya. Analisis risiko per kurir/rute memerlukan
metadata operasional yang belum tersedia. Proporsi kelas dataset tidak dapat
langsung dianggap sebagai tingkat kerusakan pengiriman nyata.

## Reproduksi dan penyimpanan

Jalankan sel kode pada notebook ini secara berurutan. Untuk Colab, pilih sumber data ZIP atau Google Drive pada konfigurasi. Dependensi dipasang jika belum tersedia.

Output lokal di `artifacts/dataset_audit/`:
- `summary.json`: ringkasan angka audit.
- `training_image_attributes.csv`: bukti atribut dan status per gambar.
- `file_sha256_duplicates.csv`, `pixel_sha256_duplicates.csv`: bukti duplikasi.
- `baseline_validation_duplicates_in_train.csv`: bukti overlap validation.

Folder dataset, arsip ZIP bernama dataset, dan seluruh `artifacts/` masuk
`.gitignore`. Dataset tidak terlacak Git dan tidak ditemukan dalam riwayat
lokal yang tersedia untuk path tersebut. Audit ini tidak melakukan commit/push.


## Reproduksi audit — jalankan dari sini

Gunakan CPU. Pilih `DATA_SOURCE`: `auto` memakai folder lokal atau meminta unggahan ZIP di Colab; `folder` memakai `DATA_PATH`; `drive` mount Google Drive lalu memakai `DATA_PATH`; `zip` memakai `ZIP_PATH` atau meminta unggahan.

Folder dataset harus berisi `Train_Set`, `Test_Set`, dan `sample_submission.csv`. Gunakan ZIP resmi panitia. Hanya piksel training dibaca; test diperiksa sebatas nama file. Output rinci berada di `artifacts/dataset_audit/` dan tidak perlu di-commit.

Angka overlap split mengikuti urutan discovery filesystem runtime, sehingga dapat berbeda dari laporan historis. Dataset asli tidak diubah.

In [ ]:
import importlib.util
import subprocess
import sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
            'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
import platform
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import f1_score
import importlib.metadata

SEED = 42
CLASSES = ['Aman', 'Rusak', 'Lainnya']
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IN_COLAB = 'google.colab' in sys.modules
np.random.seed(SEED)
versions = {p: importlib.metadata.version(p) for p in required.values()}
print('Python:', platform.python_version(), '| Colab:', IN_COLAB)
display(pd.Series(versions, name='versi runtime'))


In [ ]:
# Ubah hanya konfigurasi ini untuk sumber data Anda.
DATA_SOURCE = 'auto'  # auto | folder | drive | zip
DATA_PATH = ''        # contoh: /content/drive/MyDrive/DATASCAPE/dataset
ZIP_PATH = ''         # contoh: /content/drive/MyDrive/DATASCAPE/dataset.zip
WORKERS = 4
OUTPUT_DIR = Path.cwd() / 'artifacts' / 'dataset_audit'
EXTRACT_DIR = Path.cwd() / 'data' / 'datascape_extracted'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_SOURCE in {'auto', 'folder', 'drive', 'zip'}
assert 1 <= WORKERS <= 8


In [ ]:
def extract_official_zip(archive):
    # Validate members before extraction; refuse links/path traversal.
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    destination = EXTRACT_DIR.resolve()
    with zipfile.ZipFile(archive) as zf:
        for entry in zf.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('ZIP memuat path di luar folder ekstraksi.')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP berisi symbolic link; gunakan arsip resmi biasa.')
            if target.exists() and not entry.is_dir():
                raise FileExistsError(f'Ekstraksi tidak menimpa file lama: {target}. Gunakan DATA_SOURCE=folder.')
        zf.extractall(destination)
    return destination


def upload_zip():
    if not IN_COLAB:
        raise FileNotFoundError('Isi DATA_PATH atau ZIP_PATH untuk eksekusi lokal.')
    from google.colab import files
    print('Unggah satu ZIP resmi dataset. Untuk data besar, gunakan opsi drive.')
    uploaded = files.upload()
    names = list(uploaded)
    del uploaded
    if len(names) != 1 or not names[0].lower().endswith('.zip'):
        raise ValueError('Diperlukan tepat satu file ZIP.')
    return Path(names[0])


if DATA_SOURCE == 'drive':
    if not IN_COLAB:
        raise RuntimeError('Opsi drive ditujukan untuk Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    if not DATA_PATH:
        raise ValueError('Isi DATA_PATH ke folder atau ZIP di Google Drive.')
    base = Path(DATA_PATH).expanduser()
elif DATA_SOURCE == 'folder':
    if not DATA_PATH:
        raise ValueError('Isi DATA_PATH terlebih dahulu.')
    base = Path(DATA_PATH).expanduser()
elif DATA_SOURCE == 'zip':
    base = Path(ZIP_PATH).expanduser() if ZIP_PATH else upload_zip()
else:
    candidates = [Path(DATA_PATH).expanduser()] if DATA_PATH else [
        Path.cwd() / 'datascape-2026-data-mining-competition', EXTRACT_DIR]
    base = next((p for p in candidates if p.exists()), None)
    if base is None:
        base = Path(ZIP_PATH).expanduser() if ZIP_PATH else upload_zip()

if base.is_file() and base.suffix.lower() == '.zip':
    base = extract_official_zip(base)
if not base.is_dir():
    raise FileNotFoundError(f'Folder dataset tidak ditemukan: {base}')

# A wrapper directory inside the ZIP is supported; ambiguous roots are rejected.
roots = [base] if (base / 'Train_Set').is_dir() else [p.parent for p in base.rglob('Train_Set') if p.is_dir()]
roots = [p for p in roots if (p / 'Test_Set').is_dir() and (p / 'sample_submission.csv').is_file()]
if len(roots) != 1:
    raise ValueError(f'Diharapkan satu root dataset lengkap, ditemukan {len(roots)}. Isi DATA_PATH lebih spesifik.')
DATA_ROOT = roots[0].resolve()
print('Dataset:', DATA_ROOT)
print('Output lokal:', OUTPUT_DIR)


In [ ]:
# Konfigurasi audit mengikuti dataset yang ditemukan pada sel sebelumnya.
DATA = DATA_ROOT
ROOT = DATA_ROOT.parent
OUT = OUTPUT_DIR


## Fungsi audit
Definisi fungsi saja; audit dijalankan pada sel terakhir.

In [ ]:
def inspect(path):
    row = {'filepath': str(path.relative_to(ROOT)), 'filename': path.name,
           'label': path.relative_to(DATA / 'Train_Set').parts[0]}
    try:
        raw = path.read_bytes()
        row.update(file_bytes=len(raw), file_sha256=hashlib.sha256(raw).hexdigest())
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            row.update(format=im.format, mode=im.mode, exif_present=bool(im.getexif()))
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            w, h = rgb.size
            pixel_hash = hashlib.sha256(f'{w}x{h}:RGB:'.encode() + rgb.tobytes()).hexdigest()
            gray = np.asarray(rgb.convert('L'), dtype=np.float32) / 255
            small = np.asarray(rgb.resize((128, 128)).convert('L'), dtype=np.float32) / 255
            lap = (small[:-2, 1:-1] + small[2:, 1:-1] + small[1:-1, :-2]
                   + small[1:-1, 2:] - 4 * small[1:-1, 1:-1])
            row.update(width=w, height=h, aspect_ratio=w/h,
                       brightness=float(gray.mean()), contrast=float(gray.std()),
                       dark_pixel_fraction=float((gray <= 0.05).mean()),
                       bright_pixel_fraction=float((gray >= 0.95).mean()),
                       sharpness_proxy=float(lap.var()), pixel_sha256=pixel_hash,
                       error='')
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    return row

In [ ]:
def duplicate_stats(df, key):
    grouped = df.groupby(key).agg(n=('filename', 'size'), labels=('label', 'nunique'))
    dup = grouped[grouped.n > 1]
    return {'groups': int(len(dup)), 'images_in_groups': int(dup.n.sum()),
            'redundant_copies': int((dup.n - 1).sum()),
            'cross_label_groups': int((dup.labels > 1).sum())}

In [ ]:
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    paths = sorted(p for p in (DATA / 'Train_Set').rglob('*') if p.suffix.lower() in EXT)
    test = sorted(p for p in (DATA / 'Test_Set').rglob('*') if p.suffix.lower() in EXT)
    rows = []
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for i, row in enumerate(pool.map(inspect, paths), 1):
            rows.append(row)
            if i % 1000 == 0:
                print(f'Audited {i}/{len(paths)} training images', flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'training_image_attributes.csv', index=False)
    good = df[df.error.eq('')].copy()
    submission = pd.read_csv(DATA / 'sample_submission.csv', dtype=str)
    names = [p.name for p in test]
    metrics = ['width', 'height', 'aspect_ratio', 'brightness', 'contrast', 'sharpness_proxy']
    summary = {
        'scope': 'Full training image audit; test filenames only, no test pixel inspection.',
        'training_images': len(paths), 'test_images': len(test),
        'class_counts': df.label.value_counts().to_dict(),
        'read_errors': int(df.error.ne('').sum()),
        'formats': good.format.value_counts().to_dict(),
        'modes': good['mode'].value_counts().to_dict(),
        'exif_present': int(good.exif_present.sum()),
        'submission_columns': list(submission.columns),
        'submission_rows': len(submission),
        'submission_placeholder_counts': submission.label.value_counts().to_dict(),
        'submission_duplicate_ids': int(submission.filename.duplicated().sum()),
        'test_duplicate_filenames': len(names)-len(set(names)),
        'submission_missing_files': sorted(set(submission.filename)-set(names)),
        'test_files_not_in_submission': sorted(set(names)-set(submission.filename)),
        'file_duplicates': duplicate_stats(good, 'file_sha256'),
        'pixel_duplicates': duplicate_stats(good, 'pixel_sha256'),
        'numeric_summary': good[metrics].describe().to_dict(),
        'class_medians': good.groupby('label')[metrics].median().to_dict(orient='index'),
        'quality_flags': {
            'definition': 'Exploratory thresholds, not validated defect labels.',
            'mean_brightness_below_0.2': int((good.brightness < .2).sum()),
            'mean_brightness_above_0.8': int((good.brightness > .8).sum()),
            'short_side_below_224': int((good[['width','height']].min(axis=1) < 224).sum()),
            'aspect_ratio_outside_0.5_to_2': int(((good.aspect_ratio < .5) | (good.aspect_ratio > 2)).sum())}
    }
    for key in ['file_sha256', 'pixel_sha256']:
        good[good.duplicated(key, keep=False)].sort_values(key).to_csv(OUT / f'{key}_duplicates.csv', index=False)
    summary['baseline_split_audit'] = audit_baseline_split(good)
    (OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2), flush=True)

In [ ]:
def audit_baseline_split(good):
    ordered_paths = []
    for folder in (DATA / 'Train_Set').iterdir():
        if folder.is_dir() and folder.name.lower() in {'aman', 'rusak', 'lainnya'}:
            ordered_paths.extend(str(p.relative_to(ROOT)) for p in folder.glob('*')
                                 if p.suffix.lower() in EXT)
    indexed = good.set_index('filepath', drop=False)
    ordered = indexed.loc[[p for p in ordered_paths if p in indexed.index]]
    train, val = train_test_split(ordered, test_size=.2, stratify=ordered.label, random_state=42)
    overlap = set(train.pixel_sha256) & set(val.pixel_sha256)
    leaked = val[val.pixel_sha256.isin(overlap)]
    leaked.to_csv(OUT / 'baseline_validation_duplicates_in_train.csv', index=False)
    return {'method': 'Notebook discovery order on current filesystem, stratified 80/20, seed 42; unreadable files excluded.',
            'train_images': len(train), 'validation_images': len(val),
            'shared_pixel_hash_groups': len(overlap),
            'validation_images_identical_to_training': len(leaked),
            'validation_fraction_identical_to_training': len(leaked)/len(val)}

## Jalankan dan simpan bukti
Setelah meninjau hasil, gunakan lembar keputusan pada notebook data understanding. Sebelum commit, bersihkan semua output notebook agar dataset atau gambar tidak ikut tersimpan.

In [ ]:
main()